In [ ]:
import os
import sys

import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from semantic_ai_agent.cache import (
    CacheEvaluator,
    CacheResult,
    CacheResults,
    PerfEval,
    SemanticCacheWrapper,
    config,
    load_openai_key,
    try_connect_to_redis,
)

## 1. Environment & Redis Connection

Load environment variables and verify that Redis is reachable.

Make sure Redis is running:
```bash
docker compose up -d
```

In [ ]:
load_dotenv()
load_openai_key()

print("Config:", config)
r = try_connect_to_redis(config["redis_url"])

## 2. Initialize the Semantic Cache

Create a `SemanticCacheWrapper` from the project config.

In [ ]:
cache = SemanticCacheWrapper.from_config(config)
print(f"Cache '{config['cache_name']}' ready (threshold={config['distance_threshold']})")

## 3. Hydrate the Cache

Populate the cache with sample question-answer pairs.
Replace this with your own domain data.

In [ ]:
sample_data = pd.DataFrame({
    "question": [
        "What is your return policy?",
        "How do I track my order?",
        "Do you offer international shipping?",
        "What payment methods do you accept?",
        "How can I contact customer support?",
    ],
    "answer": [
        "You can return any item within 30 days of purchase for a full refund.",
        "You can track your order using the tracking link sent to your email after shipping.",
        "Yes, we ship to over 50 countries worldwide. Shipping rates vary by destination.",
        "We accept credit cards (Visa, MasterCard, Amex), PayPal, and Apple Pay.",
        "You can reach us via email at support@example.com or call 1-800-555-0123.",
    ],
})

cache.hydrate_from_df(sample_data, q_col="question", a_col="answer")
print(f"Cache hydrated with {len(sample_data)} entries")
sample_data

## 4. Query the Cache

Test with semantically similar queries that don't match the exact cached text.

In [ ]:
result = cache.check("Can I get a refund?")
print(f"Query: '{result.query}'")
for match in result.matches:
    print(f"  Match: '{match.prompt}'")
    print(f"  Response: '{match.response}'")
    print(f"  Distance: {match.vector_distance:.4f}")
    print(f"  Cosine Similarity: {match.cosine_similarity:.4f}")

In [ ]:
test_queries = [
    "How do I return a product?",
    "Where is my package?",
    "Do you ship outside the US?",
    "What credit cards can I use?",
    "How do I get in touch with you?",
    "What is the meaning of life?",
]

results = cache.check_many(test_queries, show_progress=True)

for r in results:
    hit = "HIT" if r.matches else "MISS"
    distance = f"{r.matches[0].vector_distance:.4f}" if r.matches else "N/A"
    print(f"[{hit}] '{r.query}' -> distance={distance}")

## 5. Evaluate Cache Effectiveness

Use `CacheEvaluator` to measure precision, recall, and F1.

In [ ]:
true_labels = [True, True, True, True, True, False]

evaluator = CacheEvaluator(true_labels, results)
metrics = evaluator.get_metrics(distance_threshold=config["distance_threshold"])

print(f"Cache Hit Rate: {metrics['cache_hit_rate']:.2%}")
print(f"Precision:      {metrics['precision']:.2%}")
print(f"Recall:         {metrics['recall']:.2%}")
print(f"F1 Score:       {metrics['f1_score']:.2%}")
print(f"Accuracy:       {metrics['accuracy']:.2%}")

In [ ]:
evaluator.matches_df()

## 6. Performance Tracking

Use `PerfEval` to measure latency across cache lookups.

In [ ]:
perf = PerfEval()
perf.set_total_queries(len(test_queries))

with perf:
    for query in test_queries:
        perf.start()
        result = cache.check(query)
        label = "cache_hit" if result.matches else "cache_miss"
        perf.tick(label)

print(perf.summary(labels=["cache_hit", "cache_miss"]))

## 7. Threshold Sweep

Find the optimal distance threshold by sweeping across a range.

In [ ]:
full_results = cache.check_many(
    test_queries, distance_threshold=1.0, show_progress=True
)

full_evaluator = CacheEvaluator.from_full_retrieval(true_labels, full_results)
sweep = full_evaluator.sweep_thresholds(metric_to_maximize="f1_score")

print(f"Best threshold for F1: {sweep['best_threshold']:.3f}")

## 8. Cleanup

Clear the cache when done experimenting.

In [ ]:
# cache.clear()
# print("Cache cleared")